In [45]:
import os
import re
import requests
import pandas as pd

MASTER_CSV = "playerss.csv"


In [46]:
SITES = {
    #1
    "sju": (
        "Saint Joseph's University",
        "https://sjuhawks.com/sports/baseball/roster/2026",
    ),
    #2
    "villanova": (
         "Villanova University",
         "https://villanova.com/sports/baseball/roster/2026"
     ),
    #3
     "upenn": (
         "University of Pennsylvania",
         "https://pennathletics.com/sports/baseball/roster"
     ),

    #4
    "brown":(
        "Brown University",
        "https://brownbears.com/sports/baseball/roster"
    ),

    #5
    "fordham":(
        "Fordham University",
        "https://fordhamsports.com/sports/baseball/roster"
    ),

    #6
    "hofstra":(
        "Hofstra University",
        "https://gohofstra.com/sports/baseball/roster"
    ),

    #7
    "butler":(
        "Butler University", 
        "https://butlersports.com/sports/baseball/roster"
    ),

    #8
    "iona":(
        "Iona University",
        "https://towsontigers.com/sports/baseball/roster"
    ),

    #9
    "cornell":(
        "Cornell University", 
        "https://cornellbigred.com/sports/baseball/roster"
    ),

    #10
    "boston":(
        "Boston College",
        "https://bceagles.com/sports/baseball/roster"
    ),

    #11
    "princeton":(
        "Princeton University",
        "https://goprincetontigers.com/sports/baseball/roster"
    ),
    
    #12
    "georgetown":(
        "Georgetown University",
        "https://guhoyas.com/sports/baseball/roster/2025"
    ),

    #13
    "north_carolina":(
        "North Carolina State University",
        "https://gopack.com/sports/baseball/roster"
    ),

    #14
    "georgia":(
        "University of Georgia",
        "https://georgiadogs.com/sports/baseball/roster"
    ),

    #15
    "elon":(
        "Elon University",
        "https://elonphoenix.com/sports/baseball/roster"
    ),
    
    #16
    "monmouth":(
        "Monmouth University",
        "https://monmouthhawks.com/sports/baseball/roster"
    ),
    
    #17
    "ball":(
        "Ball State University",
        "https://ballstatesports.com/sports/baseball/roster"
    ),

    #18
    "bucknel":(
        " Bucknell University",
        "https://bucknellbison.com/sports/baseball/roster"
    ),

    #19
    "rudgers":(
        "Rutgers University",
        "https://scarletknights.com/sports/baseball/roster"
    ),

    #20
    "gannon":(
        "Gannon University",
        "https://gannonsports.com/sports/baseball/roster"
    ),
    
}

In [47]:
def process_site(name_key: str) -> pd.DataFrame:
    name_key = name_key.lower().strip()
    if name_key not in SITES:
        raise ValueError(
            f"Unknown site key '{name_key}'. "
            f"Known keys are: {list(SITES.keys())}"
        )

    label, url = SITES[name_key]
    return process_website(url=url, label=label)

In [48]:
def fetch_html(url: str) -> str:
    """
    Download HTML for the given URL.
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; BaseballSchemaBot/1.0)"
    }
    resp = requests.get(url, headers=headers, timeout=20)
    resp.raise_for_status()
    return resp.text


def pick_main_table(tables: list[pd.DataFrame]) -> pd.DataFrame | None:
    """
    Choose the 'main' table by size (rows * columns).
    """
    if not tables:
        return None

    best_df = None
    best_size = -1

    for df in tables:
        rows, cols = df.shape
        if rows < 3 or cols < 2:
            continue
        size = rows * cols
        if size > best_size:
            best_size = size
            best_df = df

    return best_df


def extract_main_table(url: str) -> pd.DataFrame:
    """
    Fetch HTML, parse tables, pick the main one, return DataFrame.
    """
    html = fetch_html(url)

    try:
        tables = pd.read_html(html)
    except ValueError:
        raise RuntimeError("No HTML tables found on this page.")

    main_df = pick_main_table(tables)
    if main_df is None:
        raise RuntimeError("Could not find a suitable data table.")

    main_df.columns = [str(c).strip() for c in main_df.columns]
    return main_df


In [49]:
def load_master() -> pd.DataFrame:
    """
    Load master CSV if it exists, else return empty DataFrame.
    """
    if os.path.exists(MASTER_CSV):
        df = pd.read_csv(MASTER_CSV)
        print(f"Loaded master with {len(df)} rows, {len(df.columns)} columns.")
        return df
    else:
        print("No master file found. Starting new master.")
        return pd.DataFrame()


In [50]:
def _clean_str(x):
    if pd.isna(x):
        return x
    x = str(x).strip().strip('"').strip("'")
    x = re.sub(r"\s+", " ", x)
    return x if x != "" else pd.NA


def apply_split_rules(df: pd.DataFrame) -> pd.DataFrame:
    """
    Split 'Hometown / High School'-type columns into hometown + high_school.
    """
    out = df.copy()

    combined_candidates = [
        "Hometown / High School",
        "Hometown/ High School",
        "Hometown /High School",
        "Hometown/High School",
        "Hometown / Highschool",
        "Hometown/Highschool",
    ]

    combined_col = next((c for c in combined_candidates if c in out.columns), None)

    # Ensure target columns exist
    if "hometown" not in out.columns:
        out["hometown"] = pd.NA
    if "high_school" not in out.columns:
        out["high_school"] = pd.NA

    if combined_col:
        parts = out[combined_col].astype(str).str.split(r"\s*/\s*", n=1, expand=True)
        hometown_parsed = parts[0].map(_clean_str)
        highschool_parsed = (
            parts[1].map(_clean_str) if parts.shape[1] > 1 else pd.Series([pd.NA] * len(out))
        )

        out["hometown"] = out["hometown"].fillna(hometown_parsed)
        out["high_school"] = out["high_school"].fillna(highschool_parsed)

        out = out.drop(columns=[combined_col])

    # clean the split columns
    for col in ("hometown", "high_school"):
        if col in out.columns:
            out[col] = out[col].map(_clean_str)

    return out


In [51]:
# Raw header -> canonical header
CANON_MAP = {
    # jersey / roster number
    "No.": "jersey",
    "#": "jersey",
    "-": "jersey",
    "Unnamed: 1": "jersey",
    "No" : "jersey",
    "Number" : "jersey",


    # names
    "Name": "name",
    "Full Name": "name",

    # year/class
    "Year": "year",
    "Academic Year": "year",
    "Yr.": "year",
    "Class" : "year",
    "Cl." : "year",

    # position
    "Position": "position",
    "Pos.": "position",
    "Pos" :"position",

    # bats/throws
    "Bats/Throws": "bats_throws",
    "B/T": "bats_throws",

    # height/weight
    "Height": "height",
    "Ht.": "height",
    "Ht": "height",
    "Weight": "weight",
    "Wt.": "weight",
    "Wt": "weight",

    # background
    "Hometown": "hometown",
    "High School": "high_school",
    "High School (Last School)" : "high_school",
    "Previous School": "previous_school",
    "Previous College": "previous_school",
    "Last School" : "previous_school",
    "High School/Previous" : "high_school",
    # Hometown / High School is handled by split step; we don't map it directly

    # extras
    "Twitter": "twitter",
    "Headshot": "headshot",
    "Major": "major",

    # meta
    "source_website": "source_website",
}


def apply_canonical_mapping(df: pd.DataFrame) -> pd.DataFrame:
    
    out = df.copy()

    rename_now = {c: CANON_MAP[c] for c in out.columns if c in CANON_MAP}
    out = out.rename(columns=rename_now)

    # merge duplicate columns by name (take first non-null)
    out = out.groupby(axis=1, level=0).first()

    return out


In [52]:
def process_website(url: str, label: str) -> pd.DataFrame:
    
    print(f"\n=== Processing: {label} ===")
    df_raw = extract_main_table(url)
    print("Raw shape:", df_raw.shape)
    print("Raw columns:", list(df_raw.columns))

    # you currently store label, keeping that as-is
    df_raw["university"] = label

    df_split = apply_split_rules(df_raw)
    df_mapped = apply_canonical_mapping(df_split)

    if "height" in df_mapped.columns:
        df_mapped["height"] = df_mapped["height"].map(normalize_height_simple)
        
    df_typed = coerce_types(df_mapped)
    df_final = reorder_columns(df_typed)

    # 1) Remove any rows that contain "skip ad" anywhere (case-insensitive)
    mask_bad = df_final.astype(str).apply(
    lambda col: col.str.contains(r"skip ad|team impact", case=False, na=False)
        ).any(axis=1)

    n_bad = int(mask_bad.sum())
    if n_bad > 0:
        print(f"Removing {n_bad} rows containing 'skip ad' or 'Team Impact' for {label}.")
        df_final = df_final[~mask_bad]


    # 2) Load master and concatenate
    master = load_master()
    combined = pd.concat([master, df_final], ignore_index=True, sort=False)

    # 3) Drop twitter + instagram columns from the combined master
    cols_to_drop = [
        c for c in combined.columns
        if c.strip().lower() in {"twitter", "instagram","c"}
    ]
    if cols_to_drop:
        print("Dropping columns from master:", cols_to_drop)
        combined = combined.drop(columns=cols_to_drop)

    # 4) Save updated master
    combined.to_csv(MASTER_CSV, index=False)
    print(f"Master now: {len(combined)} rows, {len(combined.columns)} columns.")
    print("Columns:", list(combined.columns))

    return combined


In [53]:
import re
import pandas as pd

def normalize_height_simple(val):
    if pd.isna(val):
        return val

    s = str(val).strip()

    # Normalize different quote variants
    s = s.replace("’", "'").replace("`", "'")
    # Remove double quotes and double single-quotes
    s = s.replace('"', "").replace("''", "")
    # Collapse multiple spaces
    s = re.sub(r"\s+", " ", s)

    # Case 1: "6-0" -> "6'0"
    if "-" in s:
        feet, inches = s.split("-", 1)
        feet = feet.strip()
        inches = inches.strip()

        # keep only digits from each part (just in case)
        m_feet = re.search(r"\d+", feet)
        m_inch = re.search(r"\d+", inches)
        if m_feet and m_inch:
            return f"{m_feet.group()}'{m_inch.group()}"
        else:
            return s  # fallback

    # Case 2: "5'11", "5' 11", "6'0", "6' 0"
    m = re.search(r"(\d+)\s*'\s*(\d+)", s)
    if m:
        feet, inches = m.groups()
        return f"{feet}'{inches}"

    # Fallback: leave it unchanged if we don't recognize pattern
    return s


In [ ]:
def coerce_types(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # jersey: integer-like
    if "jersey" in out.columns:
        out["jersey"] = (
            pd.to_numeric(out["jersey"], errors="coerce")
            .astype("Int64")  # nullable integer, so NaN is allowed
        )

    # weight: normal numeric (float is fine)
    if "weight" in out.columns:
        out["weight"] = pd.to_numeric(out["weight"], errors="coerce")

    return out


def reorder_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Put the most important columns in a nice order; leave all others after.
    """
    preferred_order = [
        "jersey", "name", "year", "position", "bats_throws",
        "height", "weight", "hometown", "high_school", "previous_school",
        "twitter", "headshot", "major", "university",
    ]
    ordered = [c for c in preferred_order if c in df.columns]
    rest = [c for c in df.columns if c not in ordered]
    return df[ordered + rest]


In [ ]:
all_dfs = {}  # store each site's dataframe

for key in SITES.keys():
    print(f"Processing: {key}")
    try:
        df = process_site(key)
        all_dfs[key] = df
    except Exception as e:
        print(f" X- Error while processing {key}: {e}")


Processing: sju

=== Processing: Saint Joseph's University ===


/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_59459/1904520584.py:42: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)
/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_59459/3293183902.py:68: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  out = out.groupby(axis=1, level=0).first()


Raw shape: (35, 9)
Raw columns: ['#', 'Name', 'Year', 'Position', 'Bats/Throws', 'Height', 'Weight', 'Hometown / High School', 'Previous School']
No master file found. Starting new master.
Master now: 35 rows, 11 columns.
Columns: ['jersey', 'name', 'year', 'position', 'bats_throws', 'height', 'weight', 'hometown', 'high_school', 'previous_school', 'university']
Processing: villanova

=== Processing: Villanova University ===


/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_59459/1904520584.py:42: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)
/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_59459/3293183902.py:68: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  out = out.groupby(axis=1, level=0).first()


Raw shape: (33, 9)
Raw columns: ['#', 'Full Name', 'Academic Year', 'B/T', 'Pos.', 'Ht.', 'Wt.', 'Hometown / High School', 'Previous School']
Loaded master with 35 rows, 11 columns.
Master now: 68 rows, 11 columns.
Columns: ['jersey', 'name', 'year', 'position', 'bats_throws', 'height', 'weight', 'hometown', 'high_school', 'previous_school', 'university']
Processing: upenn

=== Processing: University of Pennsylvania ===


...

/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_59459/1904520584.py:42: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)
/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_59459/3293183902.py:68: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  out = out.groupby(axis=1, level=0).first()


Raw shape: (39, 9)
Raw columns: ['#', 'Full Name', 'B/T', 'Pos.', 'Ht.', 'Wt.', 'Year', 'Hometown', 'High School']
Removing 2 rows containing 'skip ad' or 'Team Impact' for  Bucknell University.
Loaded master with 630 rows, 12 columns.
Master now: 667 rows, 12 columns.
Columns: ['jersey', 'name', 'year', 'position', 'bats_throws', 'height', 'weight', 'hometown', 'high_school', 'previous_school', 'university', 'major']
Processing: rudgers

=== Processing: Rutgers University ===


/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_59459/1904520584.py:42: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)
/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_59459/3293183902.py:68: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  out = out.groupby(axis=1, level=0).first()


Raw shape: (39, 9)
Raw columns: ['#', 'Full Name', 'Pos.', 'Yr.', 'Ht.', 'Wt.', 'B/T', 'Hometown / High School', 'Major']
Loaded master with 667 rows, 12 columns.
Master now: 706 rows, 12 columns.
Columns: ['jersey', 'name', 'year', 'position', 'bats_throws', 'height', 'weight', 'hometown', 'high_school', 'previous_school', 'university', 'major']
Processing: gannon

=== Processing: Gannon University ===
Raw shape: (46, 9)
Raw columns: ['#', 'Full Name', 'Pos.', 'Ht.', 'Wt.', 'Year', 'B/T', 'Hometown / High School', 'Previous School']
Loaded master with 706 rows, 12 columns.
Master now: 752 rows, 12 columns.
Columns: ['jersey', 'name', 'year', 'position', 'bats_throws', 'height', 'weight', 'hometown', 'high_school', 'previous_school', 'university', 'major']


/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_59459/1904520584.py:42: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)
/var/folders/bf/sbry5wts551d8l1lfbz4y6br0000gn/T/ipykernel_59459/3293183902.py:68: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  out = out.groupby(axis=1, level=0).first()
